# Stamos S-LS-60 electronic load

Python notebook example for the `Sls60` pyserial driver. The driver supports the serial interface and UDP control (for devices on a `192.*` address), measurements, operating modes, and input on/off control.

Install pyserial if needed: `pip install pyserial`.

**Safety:** Connect an appropriate source/load, check current and voltage limits, and supervise the setup. The notebook initially leaves the electronic load input off. Only run the input-enable cell when you are ready to apply the configured load.

In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import Skripte

In [2]:
import time

from Skripte.Sls60 import Sls60

# Set either a serial port such as "COM28" or a device IP such as "192.168.178.31".
DEVICE = "192.168.178.31"
TEST_CURRENT_A = 0.01

## Connect and inspect

The connection call checks the device identification response. It returns `False` if the device does not respond.

In [3]:
load = Sls60(debug_level=1)
connected = load.connect(DEVICE)
if not connected:
    raise ConnectionError(f"Could not identify the S-LS-60 at {DEVICE}")

print("Device:", load.get_version())
print("Function:", load.get_function())
print("Input enabled:", load.get_input())

Device: S-LS-60 V3.10 SN:00013530
Function: CC
Input enabled: False


## Read measurements and configure CC mode

This reads voltage, current, and power with the input off, selects constant-current mode, and sets a conservative test current. It does not enable the load input.

In [4]:
load.set_input(False)
voltage_off = load.measure_voltage()
current_off = load.measure_current()
power_off = load.measure_power()
print(f"Input off: {voltage_off:.4f} V, {current_off:.4f} A, {power_off:.4f} W")

load.set_function("CURR")
load.set_cc_current(TEST_CURRENT_A)
print("Mode:", load.get_function())
print(f"Configured current: {load.get_cc_current():.4f} A")

Input off: 5.3279 V, 0.0000 A, 0.0000 W
Mode: CC
Configured current: 0.0100 A


## Optional load test

Run this cell only after verifying the source and current limit. It enables the input briefly, reads the measurements, and switches the input off in a `finally` block.

In [5]:
load.set_input(True)
for i in range(5):
    time.sleep(0.5)
    voltage_on = load.measure_voltage()
    current_on = load.measure_current()
    power_on = load.measure_power()
    print(f"Input on: {voltage_on:.4f} V, {current_on:.4f} A, {power_on:.4f} W")
load.set_input(False)

Input on: 4.4208 V, 0.0089 A, 0.0393 W
Input on: 1.0886 V, 0.0089 A, 0.0096 W
Input on: 0.4580 V, 0.0089 A, 0.0040 W
Input on: 0.0000 V, 0.0073 A, 0.0000 W
Input on: 0.0000 V, 0.0048 A, 0.0000 W


The driver also provides `set_cv_voltage` / `get_cv_voltage`, `set_cr_resistance` / `get_cr_resistance`, and `set_cp_power` / `get_cp_power`. Always choose the matching function before changing its setpoint. Run the cleanup cell when you are finished.

In [ ]:
load.set_input(False)
load.disconnect()
print("Input disabled; disconnected.")